# 03 - Model Training: Unified Character CNN (Numbers & Letters)

This notebook defines, compiles, and trains the Convolutional Neural Network (CNN) across all 47 character classes:
1. Loading preprocessed features from `models/character_features.npz`.
2. Defining the multi-layer CNN architecture with Batch Normalization, Dropout, and 256 dense units.
3. Compiling with Adam optimizer and Sparse Categorical Crossentropy.
4. Setting up training callbacks (`EarlyStopping`, `ReduceLROnPlateau`).
5. Running the training loop and saving weights to `models/handwritten_cnn.keras`.
6. Exporting training curves to `reports/training_curves.png`.


In [ ]:
# Import libraries
import os
import sys
import numpy as np
import matplotlib.pyplot as plt
import tensorflow as tf

sys.path.append(os.path.abspath(".."))
from src.preprocessing import load_preprocessed_data, load_label_mapping
from src.models import build_cnn_model, compile_model, save_model
from src.train import plot_training_curves

print("TensorFlow:", tf.__version__)


## 1. Load Preprocessed Data


In [ ]:
features_path = os.path.abspath("../models/character_features.npz")
x_train, y_train, x_test, y_test = load_preprocessed_data(features_path)
label_map = load_label_mapping(os.path.abspath("../models/label_mapping.json"))
num_classes = len(np.unique(y_train))
print(f"Training on {num_classes} character classes (Numbers 0-9 & Letters A-Z).")


## 2. Define CNN Architecture


In [ ]:
model = build_cnn_model(input_shape=(28, 28, 1), num_classes=num_classes, dropout_rate=0.25)
model = compile_model(model, learning_rate=0.001)
model.summary()


## 3. Training Callbacks & Training Loop


In [ ]:
callbacks = [
    tf.keras.callbacks.EarlyStopping(
        monitor="val_accuracy",
        patience=3,
        restore_best_weights=True,
        verbose=1
    ),
    tf.keras.callbacks.ReduceLROnPlateau(
        monitor="val_loss",
        factor=0.5,
        patience=2,
        min_lr=1e-5,
        verbose=1
    )
]

history = model.fit(
    x_train,
    y_train,
    epochs=4,
    batch_size=256,
    validation_split=0.1,
    callbacks=callbacks,
    verbose=1
)


## 4. Save Trained Model & Plot Curves


In [ ]:
model_save_path = os.path.abspath("../models/handwritten_cnn.keras")
save_model(model, model_save_path)

reports_curve_path = os.path.abspath("../reports/training_curves.png")
plot_training_curves(history, reports_curve_path)


## Summary
- CNN trained across all 47 classes (numbers and letters).
- Model saved to `models/handwritten_cnn.keras`.
- Training curves saved to `reports/training_curves.png`.
